# Manning's n Map Generation from LULC

This notebook converts a **Land Use / Land Cover (LULC) raster** into a **Manning's roughness
coefficient (n) raster**, saved as a GeoTIFF.

It supports the two LULC sources with their own lookup tables:

1. **NLCD** — USGS National Land Cover Database (30 m, USA). Class codes 11–95.
2. **ESRI Sentinel-2** — 10 m global land cover. Class codes 1–11.

**How it works:** every LULC class code is assigned a Manning's n value from a lookup table
(`code → (class name, min n, avg/default n, max n)`). The default (average) value is stamped
into the output raster; pixels with codes not in the table get a fallback value; nodata /
background pixels are set to `-9999`.

**Requirements:** `rasterio`, `numpy`, `pandas`, `matplotlib` (only for the preview plots).

The output Manning GeoTIFF has the **same grid, resolution, and CRS** as the input LULC raster.

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path

---
## 1. NLCD — Manning's n lookup table

Roughness ranges assembled from standard hydraulic references (Chow 1959; Kalyanapu et al. 2009;
Liu et al. 2019 and similar flood-modeling literature). The **Default n** column is what gets
written to the raster; Min/Max show the plausible range so you can adjust per study area.

In [ ]:
# NLCD class code -> (class name, min n, max n, default n)
NLCD_MANNING = {
    11: ("Open Water",                   0.025, 0.035, 0.030),
    12: ("Perennial Ice/Snow",           0.030, 0.050, 0.040),
    21: ("Developed, Open Space",        0.035, 0.065, 0.050),
    22: ("Developed, Low Intensity",     0.050, 0.110, 0.080),
    23: ("Developed, Medium Intensity",  0.070, 0.130, 0.100),
    24: ("Developed, High Intensity",    0.090, 0.150, 0.120),
    31: ("Barren Land",                  0.025, 0.045, 0.035),
    41: ("Deciduous Forest",             0.080, 0.120, 0.100),
    42: ("Evergreen Forest",             0.090, 0.130, 0.110),
    43: ("Mixed Forest",                 0.085, 0.125, 0.105),
    52: ("Shrub/Scrub",                  0.050, 0.090, 0.070),
    71: ("Grassland/Herbaceous",         0.030, 0.060, 0.045),
    81: ("Pasture/Hay",                  0.030, 0.060, 0.045),
    82: ("Cultivated Crops",             0.025, 0.055, 0.040),
    90: ("Woody Wetlands",               0.090, 0.150, 0.120),
    95: ("Emergent Herbaceous Wetlands", 0.060, 0.100, 0.080),
}

pd.DataFrame(
    [(c, v[0], v[1], v[2], v[3]) for c, v in NLCD_MANNING.items()],
    columns=["NLCD code", "Class name", "Min n", "Max n", "Default n"],
).set_index("NLCD code")

---
## 2. ESRI Sentinel-2 — Manning's n lookup table

ESRI 10 m Annual Land Cover class codes. Note: **Clouds (10)** has no valid roughness — those
pixels receive the fallback value. **Rangeland (11)** exists in the v3 product (2017–present),
which consolidated short grass + shrub.

In [ ]:
# Sentinel-2 (ESRI 10 m LULC) class code -> (class name, min n, max n, default n)
SENTINEL2_MANNING = {
    1:  ("Water",              0.025, 0.035, 0.030),
    2:  ("Trees",              0.080, 0.140, 0.110),
    3:  ("Grass",              0.030, 0.060, 0.045),
    4:  ("Flooded Vegetation", 0.060, 0.120, 0.090),
    5:  ("Crops",              0.025, 0.055, 0.040),
    6:  ("Scrub/Shrub",        0.050, 0.090, 0.070),
    7:  ("Built Area",         0.080, 0.140, 0.110),
    8:  ("Bare Ground",        0.025, 0.045, 0.035),
    9:  ("Snow/Ice",           0.020, 0.050, 0.035),
    10: ("Clouds",             None,  None,  None),
    11: ("Rangeland",          0.030, 0.060, 0.045),
}

pd.DataFrame(
    [(c, v[0], v[1], v[2], v[3]) for c, v in SENTINEL2_MANNING.items()],
    columns=["S2 code", "Class name", "Min n", "Max n", "Default n"],
).set_index("S2 code")

---
## 3. General conversion function

Works with **either** lookup table. Reads the LULC GeoTIFF, stamps each class's default n,
handles nodata, and writes a float32 Manning GeoTIFF on the same grid.

In [ ]:
def create_manning_from_lulc(lulc_tif, manning_out_path, mapping,
                             fallback_n=0.045, nodata_out=-9999.0):
    """
    Create a Manning's n GeoTIFF from a LULC classification GeoTIFF.

    Parameters
    ----------
    lulc_tif : str or Path
        Input LULC raster (integer class codes).
    manning_out_path : str or Path
        Output Manning's n raster (float32 GeoTIFF).
    mapping : dict
        Lookup table: {int_code: (class_name, min_n, max_n, default_n)}
        e.g. NLCD_MANNING or SENTINEL2_MANNING.
    fallback_n : float
        Value used for pixels whose class code is not in the table
        (or classes with no valid n, e.g. Clouds).
    nodata_out : float
        Nodata sentinel written to the output raster.

    Returns
    -------
    Path to the written Manning GeoTIFF.
    """
    lulc_tif = Path(lulc_tif)
    manning_out_path = Path(manning_out_path)
    manning_out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(lulc_tif) as src:
        lulc = src.read(1)
        lulc_nodata = src.nodata          # NLCD / Sentinel-2 exports often use 0
        profile = src.profile.copy()

    # Start with the fallback everywhere, then stamp each mapped class
    manning = np.full(lulc.shape, fallback_n, dtype=np.float32)
    for code, (name, n_min, n_max, n_default) in mapping.items():
        if n_default is None:             # e.g. Clouds -> keep fallback
            continue
        manning[lulc == int(code)] = n_default

    # Propagate LULC nodata / background into the output
    if lulc_nodata is not None:
        manning[lulc == lulc_nodata] = nodata_out
    else:
        manning[lulc == 0] = nodata_out   # common background sentinel

    profile.update(dtype="float32", count=1, nodata=nodata_out, driver="GTiff")
    if manning_out_path.exists():
        manning_out_path.unlink()
    with rasterio.open(manning_out_path, "w", **profile) as dst:
        dst.write(manning, 1)

    print(f"Manning's n raster written: {manning_out_path}")
    print(f"  grid: {profile['width']} x {profile['height']}  |  CRS: {profile['crs']}")
    return manning_out_path

---
## 4. Run — NLCD

Set your input/output paths below and run. The input must be an NLCD LULC GeoTIFF
(integer codes 11–95).

In [ ]:
# ------------- EDIT THESE PATHS -------------
nlcd_lulc_tif    = "path/to/your/NLCD_lulc.tif"        # input NLCD LULC raster
nlcd_manning_tif = "path/to/output/Manning_NLCD.tif"   # output Manning raster
# --------------------------------------------

create_manning_from_lulc(nlcd_lulc_tif, nlcd_manning_tif, NLCD_MANNING)

---
## 5. Run — ESRI Sentinel-2

Same as above, for a Sentinel-2 (ESRI 10 m) LULC GeoTIFF (integer codes 1–11).

In [ ]:
# ------------- EDIT THESE PATHS -------------
s2_lulc_tif    = "path/to/your/Sentinel2_lulc.tif"     # input Sentinel-2 LULC raster
s2_manning_tif = "path/to/output/Manning_S2.tif"       # output Manning raster
# --------------------------------------------

create_manning_from_lulc(s2_lulc_tif, s2_manning_tif, SENTINEL2_MANNING)

---
## 6. Preview (optional)

Plot the LULC map and the resulting Manning map side by side. Point it at whichever pair
you just generated.

In [ ]:
def preview(lulc_tif, manning_tif, title=""):
    with rasterio.open(lulc_tif) as s:
        lulc = s.read(1).astype(float)
        if s.nodata is not None:
            lulc[lulc == s.nodata] = np.nan
    with rasterio.open(manning_tif) as s:
        man = s.read(1)
        man[man == s.nodata] = np.nan

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    im0 = axes[0].imshow(lulc, cmap="tab20", interpolation="nearest")
    axes[0].set_title(f"LULC classes {title}")
    plt.colorbar(im0, ax=axes[0], shrink=0.7, label="class code")
    im1 = axes[1].imshow(man, cmap="viridis", interpolation="nearest")
    axes[1].set_title(f"Manning's n {title}")
    plt.colorbar(im1, ax=axes[1], shrink=0.7, label="n")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Example — uncomment the pair you generated:
# preview(nlcd_lulc_tif, nlcd_manning_tif, "(NLCD)")
# preview(s2_lulc_tif, s2_manning_tif, "(Sentinel-2)")

---
### Notes

- The **Default n** is what gets written; edit the table dictionaries above if your study
  needs different values (e.g. use Min or Max for a sensitivity analysis).
- The output Manning raster inherits the LULC raster's grid — if your flood model needs
  it aligned to a DEM grid, resample/snap it to the DEM afterwards.
- Values are the same tables used inside FIMsim (`core/nlcd.py`), so notebook results
  match the app's output for identical inputs.